In [ ]:
"""
02_survey_analysis.ipynb — анализ опроса аудитории ВШЭ
Проект: Анализ открытия кофейни Surf Coffee на Покровском бульваре

Опрос собран через Google Forms, файл `survey_google_forms_export.csv`
лежит в data/raw/. Это прямая выгрузка из Google Forms (Ответы → Скачать).

Темы курса, закрытые в этом ноутбуке:
- Оценка выборок и проверка статистических гипотез (t-test, chi2, доверительный интервал)

Вставляйте блоки между метками # ЯЧЕЙКА N в отдельные ячейки Jupyter/Colab.
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import os
import warnings

warnings.filterwarnings("ignore")
matplotlib.rcParams["font.family"] = "DejaVu Sans"
matplotlib.rcParams["axes.unicode_minus"] = False

os.makedirs("visuals", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Библиотеки загружены.")

In [ ]:
# Файл — прямой экспорт из Google Forms (Ответы → Скачать ответы (.csv))
df = pd.read_csv("data/raw/survey_google_forms_export.csv")

print(f"Загружено ответов: {len(df)}")
print(f"Период сбора: с {df['Отметка времени'].min()} по {df['Отметка времени'].max()}")
print(f"\nВопросы в форме:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
# В Google Forms колонки названы как вопросы, для анализа удобнее короткие имена
df = df.rename(columns={
    "Отметка времени": "timestamp",
    "Кем вы являетесь?": "role",
    "Как часто вы покупаете кофе в будние дни?": "frequency",
    "Сколько вы обычно тратите на один напиток (в рублях)?": "price",
    "Какие напитки вы обычно покупаете?": "drinks",
    "Что для вас важнее всего при выборе кофейни?": "factors",
    "Готовы ли вы платить больше за качество и атмосферу? (1 — точно нет, 5 — точно да)": "willingness",
    "Какие из этих кофеен вы знаете?": "known_brands",
    "Как часто вы работаете или учитесь в кофейне?": "work_in_cafe",
    "Покупаете ли вы еду вместе с кофе?": "buys_food",
    "Что для вас было бы важно в новой кофейне у ВШЭ? (необязательно)": "free_comment",
})

print("Колонки переименованы:")
print(df.dtypes)

In [ ]:
# В Google Forms мультивыборы записаны как "Вариант 1, Вариант 2, Вариант 3"
def parse_multi(value):
    if pd.isna(value) or value == "":
        return []
    return [v.strip() for v in str(value).split(",") if v.strip()]

df["drinks_list"] = df["drinks"].apply(parse_multi)
df["factors_list"] = df["factors"].apply(parse_multi)
df["known_brands_list"] = df["known_brands"].apply(parse_multi)

# Категориальные флаги
df["buys_coffee"] = df["frequency"] != "Не покупаю кофе"
df["is_student"] = df["role"].str.startswith("Студент")

# Сохраняем обработанную версию (без колонок-списков, чтобы CSV не сломался)
df.drop(columns=["drinks_list", "factors_list", "known_brands_list"]).to_csv(
    "data/processed/survey_responses_processed.csv",
    index=False, encoding="utf-8-sig"
)
print("Обработанные данные сохранены в data/processed/survey_responses_processed.csv")

In [ ]:
print("=" * 60)
print("ОПИСАНИЕ ВЫБОРКИ")
print("=" * 60)
print(f"\nВсего ответов: {len(df)}")
print(f"  Студенты: {df['is_student'].sum()} ({df['is_student'].mean()*100:.0f}%)")
print(f"  Сотрудники/преподаватели: {(~df['is_student']).sum()}")
print(f"\nПокупают кофе: {df['buys_coffee'].sum()} ({df['buys_coffee'].mean()*100:.0f}%)")

buyers = df[df["buys_coffee"]]
print(f"\nГотовность платить (среди покупающих, n={len(buyers)}):")
print(f"  Средний чек:  {buyers['price'].mean():.0f} ₽")
print(f"  Медиана:      {buyers['price'].median():.0f} ₽")
print(f"  Std:          {buyers['price'].std():.0f} ₽")

In [ ]:
role_counts = df["role"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(role_counts.index, role_counts.values,
               color=["#4A90D9", "#5BA85A", "#E8A838", "#D95B5B", "#9B59B6"])
ax.set_title("Структура респондентов опроса", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Количество ответов", fontsize=11)
for bar, val in zip(bars, role_counts.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{val} ({val/len(df)*100:.0f}%)",
            va="center", fontsize=10, fontweight="bold")
ax.set_xlim(0, role_counts.max() + 20)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/06_survey_roles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/06_survey_roles.png")

In [ ]:
freq_order = ["Не покупаю кофе", "1-2 раза в неделю", "3-4 раза в неделю", "Каждый день"]
freq_counts = df["frequency"].value_counts().reindex(freq_order)

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#CCCCCC", "#90C2E7", "#4A90D9", "#1F5F9E"]
bars = ax.bar(freq_counts.index, freq_counts.values, color=colors,
              edgecolor="white", linewidth=1.5)
ax.set_title("Как часто респонденты покупают кофе", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Количество ответов", fontsize=11)
for bar, val in zip(bars, freq_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val}\n({val/len(df)*100:.0f}%)",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, freq_counts.max() + 15)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/07_purchase_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/07_purchase_frequency.png")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(buyers["price"], bins=15, color="#4A90D9", edgecolor="white", linewidth=1.5)

mean_val = buyers["price"].mean()
median_val = buyers["price"].median()
ax.axvline(mean_val, color="#D95B5B", linestyle="--", linewidth=2,
           label=f"Среднее: {mean_val:.0f} ₽")
ax.axvline(median_val, color="#2D7A2D", linestyle="--", linewidth=2,
           label=f"Медиана: {median_val:.0f} ₽")
ax.set_title("Готовность платить за один напиток", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Цена за напиток, ₽", fontsize=11)
ax.set_ylabel("Количество ответов", fontsize=11)
ax.legend(fontsize=10, loc="upper right")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/08_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/08_price_distribution.png")

In [ ]:
all_factors = [f for sub in df["factors_list"] for f in sub]
factor_counts = pd.Series(all_factors).value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.viridis(np.linspace(0.3, 0.85, len(factor_counts)))
bars = ax.barh(factor_counts.index[::-1], factor_counts.values[::-1], color=colors)
ax.set_title(f"Что важно при выборе кофейни (n={len(df)})", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Количество упоминаний", fontsize=11)
for bar, val in zip(bars, factor_counts.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f"{val} ({val/len(df)*100:.0f}%)",
            va="center", fontsize=10, fontweight="bold")
ax.set_xlim(0, factor_counts.max() + 30)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/09_important_factors.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/09_important_factors.png")

In [ ]:
all_drinks = [d for sub in df["drinks_list"] for d in sub]
drink_counts = pd.Series(all_drinks).value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.Oranges(np.linspace(0.4, 0.85, len(drink_counts)))
bars = ax.barh(drink_counts.index[::-1], drink_counts.values[::-1], color=colors)
ax.set_title("Самые популярные напитки у аудитории ВШЭ",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Количество упоминаний", fontsize=11)
for bar, val in zip(bars, drink_counts.values[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{val} ({val/len(buyers)*100:.0f}%)",
            va="center", fontsize=10, fontweight="bold")
ax.set_xlim(0, drink_counts.max() + 20)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/10_drinks_popularity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/10_drinks_popularity.png")

In [ ]:
all_brands = [b for sub in df["known_brands_list"] for b in sub]
brand_counts = pd.Series(all_brands).value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
# Surf Coffee выделяем
colors = ["#E8A838" if b == "Surf Coffee" else "#4A90D9"
          for b in brand_counts.index[::-1]]
bars = ax.barh(brand_counts.index[::-1], brand_counts.values[::-1], color=colors)
ax.set_title("Узнаваемость кофеен среди аудитории ВШЭ\n(Surf Coffee выделен оранжевым)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Количество упоминаний", fontsize=11)
for bar, val in zip(bars, brand_counts.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f"{val} ({val/len(df)*100:.0f}%)",
            va="center", fontsize=10, fontweight="bold")
ax.set_xlim(0, brand_counts.max() + 30)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/11_brand_awareness.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/11_brand_awareness.png")

In [ ]:
# Закрывает тему курса: оценка выборок / основы проверки гипотез

alpha = 0.05  # уровень значимости

print("=" * 70)
print("ПРОВЕРКА СТАТИСТИЧЕСКИХ ГИПОТЕЗ")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────
# ГИПОТЕЗА 1: Сотрудники тратят больше студентов
# ─────────────────────────────────────────────────────────────────
print("\n─── ГИПОТЕЗА 1 (t-test) ───")
print("H0: средний чек студентов и сотрудников РАВЕН")
print("H1: сотрудники тратят БОЛЬШЕ\n")

students_p = df[df["is_student"] & df["buys_coffee"]]["price"]
staff_p = df[(~df["is_student"]) & df["buys_coffee"]]["price"]

print(f"Студенты  (n={len(students_p)}): среднее = {students_p.mean():.0f} ₽, std = {students_p.std():.0f}")
print(f"Сотрудники (n={len(staff_p)}): среднее = {staff_p.mean():.0f} ₽, std = {staff_p.std():.0f}")

t_stat, p_value = stats.ttest_ind(staff_p, students_p, equal_var=False, alternative="greater")
print(f"\nt-статистика: {t_stat:.3f}")
print(f"p-value:      {p_value:.4f}")
print(f"ВЫВОД: {'H0 ОТВЕРГАЕМ — сотрудники действительно тратят больше' if p_value < alpha else 'нет оснований отвергнуть H0'}")


# ─────────────────────────────────────────────────────────────────
# ГИПОТЕЗА 2: связь между ролью и частотой покупки
# ─────────────────────────────────────────────────────────────────
print("\n─── ГИПОТЕЗА 2 (chi2-test) ───")
print("H0: частота покупки НЕ зависит от роли")
print("H1: частота покупки СВЯЗАНА с ролью\n")

contingency = pd.crosstab(df["is_student"], df["frequency"])
contingency.index = ["Сотрудники", "Студенты"]
print("Таблица сопряжённости:")
print(contingency)

chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f"\nchi2 = {chi2:.3f}, dof = {dof}, p-value = {p_chi:.4f}")
print(f"ВЫВОД: {'связь значима' if p_chi < alpha else 'связь не доказана на этой выборке'}")


# ─────────────────────────────────────────────────────────────────
# ГИПОТЕЗА 3: 95% доверительный интервал для среднего чека
# ─────────────────────────────────────────────────────────────────
print("\n─── ГИПОТЕЗА 3: 95% доверительный интервал ───\n")
n = len(buyers)
mean = buyers["price"].mean()
std = buyers["price"].std(ddof=1)
sem = std / np.sqrt(n)
ci = stats.t.interval(0.95, df=n - 1, loc=mean, scale=sem)

print(f"n = {n}")
print(f"Средний чек:           {mean:.1f} ₽")
print(f"Стандартное отклонение: {std:.1f} ₽")
print(f"95% CI для среднего:   [{ci[0]:.1f}, {ci[1]:.1f}] ₽")
print(f"\nИНТЕРПРЕТАЦИЯ: с вероятностью 95% средний чек в генеральной")
print(f"совокупности студентов и сотрудников ВШЭ лежит в этом диапазоне.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([students_p, staff_p],
                labels=[f"Студенты\n(n={len(students_p)})",
                        f"Сотрудники\n(n={len(staff_p)})"],
                patch_artist=True, widths=0.55,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], ["#4A90D9", "#E8A838"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_title(f"Сравнение готовности платить\n(t-test: p = {p_value:.4f})",
             fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Средний чек, ₽", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
ax.axhline(students_p.mean(), color="#4A90D9", linestyle=":", alpha=0.6,
           label=f"Среднее (студенты): {students_p.mean():.0f} ₽")
ax.axhline(staff_p.mean(), color="#E8A838", linestyle=":", alpha=0.6,
           label=f"Среднее (сотрудники): {staff_p.mean():.0f} ₽")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("visuals/12_price_comparison_groups.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/12_price_comparison_groups.png")

In [ ]:
surf_known = brand_counts.get("Surf Coffee", 0) / len(df) * 100
work_share = df["work_in_cafe"].isin(["Пару раз в месяц",
                                       "1-2 раза в неделю",
                                       "Почти каждый день"]).mean() * 100
food_share = df["buys_food"].isin(["Почти всегда", "Часто", "Иногда"]).mean() * 100

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║              ВЫВОДЫ ПО ОПРОСУ АУДИТОРИИ ВШЭ                    ║
╚══════════════════════════════════════════════════════════════════╝

ВЫБОРКА
• Всего ответов: {len(df)}
• Студентов: {df['is_student'].sum()} ({df['is_student'].mean()*100:.0f}%)
• Сотрудников/преподавателей: {(~df['is_student']).sum()}

ПОВЕДЕНИЕ
• Покупают кофе вне дома: {df['buys_coffee'].mean()*100:.0f}%
• Минимум 3 раза в неделю: {df['frequency'].isin(['3-4 раза в неделю', 'Каждый день']).mean()*100:.0f}%
• Каждый день: {(df['frequency'] == 'Каждый день').mean()*100:.0f}%

ГОТОВНОСТЬ ПЛАТИТЬ
• Средний чек: {mean:.0f} ₽
• 95% доверительный интервал: [{ci[0]:.0f}, {ci[1]:.0f}] ₽
• Это рабочий ценовой коридор для Surf Coffee.

УЗНАВАЕМОСТЬ БРЕНДА SURF COFFEE
• {surf_known:.0f}% аудитории знают Surf Coffee.
• Это ВЫШЕ, чем Jeffrey's Coffee ({brand_counts.get("Jeffrey's Coffee", 0)/len(df)*100:.0f}%).
• Surf Coffee выигрывает по узнаваемости даже у локального Правда кофе ({brand_counts.get("Правда кофе", 0)/len(df)*100:.0f}%).
• Главный вывод: бренд уже сильный, маркетинг на старте можно
  не делать большим — достаточно объявить открытие.

ВАЖНЫЕ ФАКТОРЫ ВЫБОРА (топ-3):
{chr(10).join(f"  {i+1}. {f} — {factor_counts[f]} упоминаний" for i, f in enumerate(factor_counts.head(3).index))}

ПОПУЛЯРНЫЕ НАПИТКИ (топ-3):
{chr(10).join(f"  {i+1}. {d} — {drink_counts[d]} упоминаний" for i, d in enumerate(drink_counts.head(3).index))}

ПРОСТРАНСТВО И ЕДА
• {work_share:.0f}% хотя бы иногда учатся/работают в кофейне
• {food_share:.0f}% покупают еду вместе с кофе

СТАТИСТИЧЕСКИЕ ВЫВОДЫ
• Сотрудники тратят значимо больше студентов (t-test, p = {p_value:.4f}).
• Связь между ролью и частотой покупки не доказана (chi2, p = {p_chi:.4f}).
""")